In [65]:
from typing import Any
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from Bio import SeqIO
import os
from pathlib import Path

def one_hot_encode(sequence: str,
                   alphabet: str = 'ACGT',
                   neutral_alphabet: str = 'N',
                   neutral_value: Any = 0,
                   dtype=np.float32) -> np.ndarray:
  """One-hot encode sequence."""
  def to_uint8(string):
    return np.frombuffer(string.encode('ascii'), dtype=np.uint8)
  hash_table = np.zeros((np.iinfo(np.uint8).max, len(alphabet)), dtype=dtype)
  hash_table[to_uint8(alphabet)] = np.eye(len(alphabet), dtype=dtype)
  hash_table[to_uint8(neutral_alphabet)] = neutral_value
  hash_table = hash_table.astype(dtype)
  return hash_table[to_uint8(sequence)]

def prepare_input(sequence: str):
    target_length = 196608
    if len(sequence) != target_length:
        print("input length != 196608")
        return

    # Use your encoding function
    encoded = one_hot_encode(sequence)
    
    # Add the Batch Dimension: (393216, 4) -> (1, 393216, 4)
    return encoded[np.newaxis, ...]
    
def process_fasta_to_batch(fasta_path):
    encoded_list = []
    
    # 1. Iterate through the FASTA file
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq).upper()
        
        # 2. Use your prepare_input logic
        # Note: prepare_input already adds the (1, 393216, 4) dimension
        encoded_seq = prepare_input(seq)
        
        if encoded_seq is not None:
            encoded_list.append(encoded_seq)
    
    # 3. Stack them into a single batch
    # Since prepare_input returns (1, L, 4), we use concatenate on axis 0
    if encoded_list:
        batch_array = np.concatenate(encoded_list, axis=0)
        return batch_array
    else:
        return None

In [36]:
import json

keyword = '_liver_TEST_500bp.bed'
test_dir = '/home/azstephe/liverRegression/regression_liver/data/test_splits/'

out_dir = '/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/'

for d in ['log_pos']:
    dr = os.path.join(test_dir, d)
    for file in os.listdir(dr):
        if 'newRat' in file:
            continue
        filepath = os.path.join(dr, file)
        if 'non' not in filepath and keyword in filepath:
            species = filepath.split('/')[-1].split('_')[0]
            cols = ['chrom', 'start', 'end', 'peak', 'signal']
            df = pd.read_csv(filepath, sep='\t', names=cols, index_col=False)
            df['start'] = pd.to_numeric(df['start'], errors='coerce')
            df['end'] = pd.to_numeric(df['end'], errors='coerce')
            
            extension = 196608
            df['new_start'] = df['start'] - extension + 250
            df['new_end'] = df['end'] + extension - 250

            with open('/home/azstephe/liverRegression/regression_liver/data/chrom_sizes/' + species + '_chrom_sizes.json', 'r') as f:
                chrom_map = json.load(f)

            df['max_len'] = df['chrom'].map(chrom_map)

            valid_sections = df[(df['new_start'] >= 0) & (df['new_end'] <= df['max_len'])].copy()

            print(filepath)
            print(f"Original peaks: {len(df)}")
            print(f"Peaks kept: {len(valid_sections)}")
            print(f"Peaks lost: {len(df) - len(valid_sections)}")

            out_dr = os.path.join(out_dir, d)
            if not os.path.exists(out_dr):
                os.makedirs(out_dr)
            valid_sections[['chrom', 'new_start', 'new_end', 'peak', 'signal']].to_csv(os.path.join(out_dr, file), sep='\t', index=False, header=False)


/home/azstephe/liverRegression/regression_liver/data/test_splits/log_pos/cow_liver_TEST_500bp.bed
Original peaks: 1664
Peaks kept: 1659
Peaks lost: 5
/home/azstephe/liverRegression/regression_liver/data/test_splits/log_pos/pig_liver_TEST_500bp.bed
Original peaks: 1627
Peaks kept: 1624
Peaks lost: 3
/home/azstephe/liverRegression/regression_liver/data/test_splits/log_pos/rat_liver_TEST_500bp.bed
Original peaks: 2576
Peaks kept: 2575
Peaks lost: 1
/home/azstephe/liverRegression/regression_liver/data/test_splits/log_pos/mouse_liver_TEST_500bp.bed
Original peaks: 3776
Peaks kept: 3775
Peaks lost: 1
/home/azstephe/liverRegression/regression_liver/data/test_splits/log_pos/macaque_liver_TEST_500bp.bed
Original peaks: 2624
Peaks kept: 2619
Peaks lost: 5


In [54]:
!conda install -c bioconda bedtools=2.29.2 -y

Solving environment: failed with initial frozen solve. Retrying with flexible solve.
Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 4.10.1
  latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /home/azstephe/miniconda3/envs/enformer_env

  added / updated specs:
    - bedtools=2.29.2


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bedtools-2.29.2            |       hc088bd4_0        13.8 MB  bioconda
    python-3.9.16              |       h7a1cb2a_2        25.1 MB
    xz-5.2.10                  |       h5eee18b_1         429 KB
    zlib-1.2.13                |       h5eee18b_1         111 KB
    ------------------------------------------------------------
                                           Total:        39.4 MB

The following NEW pack

In [55]:
!which bedtools

/home/azstephe/miniconda3/envs/enformer_env/bin/bedtools


In [56]:
base_dir = '/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/'
sub_dirs = ['log_pos']
# sub_dirs = []

genomes = {
    'cow': '/home/azstephe/regression_liver/data/splits/cowMouse/GCF_000003205.7_Btau_5.0.1_genomic_chromName.fna',
    'rat': '/projects/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/RatGenome/rn6.fa',
    'macaque': '/projects/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/MacaqueGenome/rheMac8.fa',
    'mouse': '/projects/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/MouseGenome/mm10.fa',
    'pig': '/data/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/200MammalsFastas/susScr3.fa'
}

for d in sub_dirs:
    dr = os.path.join(base_dir, d)
    
    if not os.path.exists(dr):
        print(f"Directory not found, skipping: {dr}")
        continue

    print(f"\nProcessing directory: {d}")
    
    for file in os.listdir(dr):
        # Only process BED or narrowPeak files
        if file.endswith('.bed') or file.endswith('.narrowPeak'):
            input_path = os.path.join(dr, file)
            species = file.split('_')[0]
            
            # Create output filename by replacing extension with .fa
            output_file = os.path.splitext(file)[0] + ".fa"
            output_path = os.path.join(dr, output_file)
            
            print(f"Extracting: {file} -> {output_file}")

            genome_fasta = genomes[species]
            # Construct the command
            cmd = [
                "bedtools", "getfasta",
                "-fi", genome_fasta,
                "-bed", input_path,
                "-fo", output_path
            ]
            
            #Run the command
            try:
                subprocess.run(cmd, check=True, capture_output=True, text=True)
            except subprocess.CalledProcessError as e:
                print(f"  Error processing {file}: {e.stderr}")


Processing directory: log_pos
Extracting: cow_liver_TEST_500bp.bed -> cow_liver_TEST_500bp.fa
Extracting: macaque_liver_TEST_500bp.bed -> macaque_liver_TEST_500bp.fa
Extracting: mouse_liver_TEST_500bp.bed -> mouse_liver_TEST_500bp.fa
Extracting: pig_liver_TEST_500bp.bed -> pig_liver_TEST_500bp.fa
Extracting: rat_liver_TEST_500bp.bed -> rat_liver_TEST_500bp.fa


In [7]:
gpu_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpu_devices))

if gpu_devices:
    print("GPU Details:", gpu_devices)
else:
    print("TensorFlow is NOT using the GPU.")

Num GPUs Available:  1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-03-26 13:24:37.775272: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2026-03-26 13:24:37.804821: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:3b:00.0 name: NVIDIA RTX A6000 computeCapability: 8.6
coreClock: 1.8GHz coreCount: 84 deviceMemorySize: 47.54GiB deviceMemoryBandwidth: 715.34GiB/s
2026-03-26 13:24:37.804872: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2026-03-26 13:24:38.126423: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2026-03-26 13:24:38.126513: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublasLt.so.11
2026-03-26 13:24:38.364926: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcufft.so.10

In [20]:
enformer_model = hub.load("https://www.kaggle.com/models/deepmind/enformer/TensorFlow2/enformer/1").model
ENFORMER_INPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/")
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")

In [8]:
for subdir in os.listdir(ENFORMER_INPUTS):
    for file.fa in subdir:
    fasta_file = ENFORMER_INPUTS / subdir / file.fa
    x = process_fasta_to_batch(fasta_file)



fasta_file = "/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test2/pig_liver_TEST_500bp.fa"
x = process_fasta_to_batch(fasta_file)

print(f"Final Batch Shape: {x.shape}")

Final Batch Shape: (293, 393216, 4)


In [57]:
target_track = 205
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")
for subdir_name in os.listdir(ENFORMER_INPUTS):
    subdir_path = ENFORMER_INPUTS / subdir_name
    
    if subdir_path.is_dir():
        # Look for all .fa or .fasta files inside that subdir
        for fasta_path in subdir_path.glob("*.fa"):
            # x = process_fasta_to_batch(str(fasta_path))
            # for i in range(len(x)):
            #     # Grab one sequence and keep batch dim: (1, 393216, 4)
            #     single_input = x[i : i+1]
                
            #     pred = enformer_model.predict_on_batch(single_input)
                
            #     # Extract target track from the mouse head
            #     track_subset = pred['mouse'][0, :, target_track]
                
            #     results.append(track_subset)
                
            #     if i % 10 == 0:
            #         print(f"Processed {i}/{len(x)}")
            
            # # 5. Convert list to a single NumPy array
            # # Final Shape: (Num_Entries, 896, 1)
            # final_output = np.array(results)
            
            outfile = ENFORMER_OUTPUTS / subdir_name / subdir_path.split("_500bp.fa")[0]  + "_enformer_{str(target_track)}_predictions.npy"
            
            # 6. Save to file
            np.save(f"/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test2/pig_liver_TEST_enformer_{str(target_track)}_predictions_test.npy", final_output)
            print("Done! Saved to enformer_subset_predictions.npy")


/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test1/pig_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test1/rat_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test1/macaque_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test1/cow_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test2/pig_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test2/macaque_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test2/cow_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_test2/rat_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_inputs/log_pos/pig_liver_TEST_500bp.fa
/home/azstephe/liverRegression/regression_liver/data/enformer_input

In [58]:
target_track = 205
ENFORMER_OUTPUTS = Path("/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/")

for subdir_name in os.listdir(ENFORMER_INPUTS):
    subdir_path = ENFORMER_INPUTS / subdir_name
    if not subdir_path.is_dir(): continue

    for fasta_path in subdir_path.glob("*.fa"):
        print(f"--- Processing: {fasta_path} ---")
        
        results = [] 
        x = process_fasta_to_batch(str(fasta_path)) 
        
        for i in range(len(x)):
            # Grab one sequence: Shape (1, 196608, 4)
            single_input = x[i : i+1]
            
            # Predict
            pred = enformer_model.predict_on_batch(single_input)
            
            track_subset = pred['mouse'][0, :, target_track]
            
            # Convert to numpy immediately to save GPU memory
            results.append(track_subset.numpy() if hasattr(track_subset, 'numpy') else track_subset)
            
            if i % 10 == 0:
                print(f"  Sequence {i}/{len(x)}")

        # 3. CONVERT TO ARRAY: Shape (Num_Peaks, 896)
        final_output = np.array(results)
        
        4. BUILD OUTPUT PATH
        Remove extension safely
        stem = fasta_path.stem.replace("_500bp", "")
        out_subdir = ENFORMER_OUTPUTS / subdir_name
        out_subdir.mkdir(parents=True, exist_ok=True)
        
        save_path = out_subdir / f"{stem}_enformer_{target_track}_preds.npy"
        np.save(save_path, final_output)


--- Processing: pig_liver_TEST_500bp.fa ---
/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test1/pig_liver_TEST_enformer_205_preds.npy
--- Processing: rat_liver_TEST_500bp.fa ---
/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test1/rat_liver_TEST_enformer_205_preds.npy
--- Processing: macaque_liver_TEST_500bp.fa ---
/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test1/macaque_liver_TEST_enformer_205_preds.npy
--- Processing: cow_liver_TEST_500bp.fa ---
/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test1/cow_liver_TEST_enformer_205_preds.npy
--- Processing: pig_liver_TEST_500bp.fa ---
/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test2/pig_liver_TEST_enformer_205_preds.npy
--- Processing: macaque_liver_TEST_500bp.fa ---
/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test2/macaque_liver_TEST_enformer_205_preds.npy
--- Processi

In [66]:
target_track = 205
results = []

print(f"Starting predictions for {len(x)} sequences...")

for i in range(len(x)):
    # Grab one sequence and keep batch dim: (1, 393216, 4)
    single_input = x[i : i+1]
    
    pred = enformer_model.predict_on_batch(single_input)
    
    # Extract target track from the mouse head
    track_subset = pred['mouse'][0, :, target_track]
    
    results.append(track_subset)
    
    if i % 10 == 0:
        print(f"Processed {i}/{len(x)}")

# 5. Convert list to a single NumPy array
# Final Shape: (Num_Entries, 896, 1)
final_output = np.array(results)

# 6. Save to file
np.save(f"/home/azstephe/liverRegression/regression_liver/data/enformer_outputs/log_test2/pig_liver_TEST_enformer_{str(target_track)}_predictions_test.npy", final_output)
print("Done! Saved to enformer_subset_predictions.npy")

Starting predictions for 293 sequences...
Processed 0/293
Processed 10/293
Processed 20/293
Processed 30/293
Processed 40/293
Processed 50/293
Processed 60/293
Processed 70/293
Processed 80/293
Processed 90/293
Processed 100/293
Processed 110/293
Processed 120/293
Processed 130/293
Processed 140/293
Processed 150/293
Processed 160/293
Processed 170/293
Processed 180/293
Processed 190/293
Processed 200/293
Processed 210/293
Processed 220/293
Processed 230/293
Processed 240/293
Processed 250/293
Processed 260/293
Processed 270/293
Processed 280/293
Processed 290/293
Done! Saved to enformer_subset_predictions.npy


In [10]:
import pandas as pd
targets_mouse = pd.read_csv('https://raw.githubusercontent.com/calico/basenji/master/manuscripts/cross2020/targets_mouse.txt', sep='\t')

# Accessing row 131 gives you the info for global track 5444
print(targets_mouse.iloc[205])

index                                                       5518
genome                                                         1
identifier                                                 UW3.1
file           /home/drk/tillage/datasets/mouse/atac/uw-atlas...
clip                                                          32
scale                                                          4
sum_stat                                                    mean
description                ATAC:Hepatocytes-clusters_3-cluster_1
Name: 205, dtype: object


In [62]:
%pip install matplotlib

  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 74.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 86.2 MB/s  0:00:00
Using cached importlib_resources-6.5.2-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 92.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 85.9 MB/s  0:00:00
  Attempting uninstall: numpy0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [pillow]
    Found existing installation: numpy 1.19.5━━━━━━━━━━━━━━━━━ 1/9 [pillow]
    Uninstalling numpy-1.19.5:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/9 [numpy]
      Successfully uninstalled numpy-1.19.5━━━━━━━━━━━━━━━━━━━ 2/9 [numpy]
   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/9 [numpy]  WARNING: Failed to remove contents in a temporary directory '/home/azstephe/miniconda3/envs/enformer_env/lib/python3.

In [63]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

ENFORMER_OUT = '/home/azstephe/liverRegression/regression_liver/data/enformer_outputs'

target_track = 205
preds_raw = np.load(f"{ENFORMER_OUT}/log_test2/pig_liver_TEST_enformer_{str(target_track)}_predictions_test.npy")

ImportError: Matplotlib requires numpy>=1.23; you have 1.19.5

In [14]:
preds_middle_4 = preds_raw[:, 446:450]

# 3. Take the average across those 4 bins
# axis=1 collapses the 4 bins into one average value
preds_avg = np.mean(preds_middle_4, axis=1).flatten()

# 3. Load the true values from the .bed file
species = "pig" # adjust as needed
bed_path = f'/home/azstephe/liverRegression/regression_liver/data/test_splits/log_test2/{species}_liver_TEST_500bp.bed'
true_values = pd.read_csv(bed_path, header=None, delim_whitespace=True).iloc[:, 4].values

# 4. Verify the 1-to-1 ratio
if len(preds_avg) != len(true_values):
    print(f"Warning: Size mismatch! Preds: {len(preds_avg)}, True: {len(true_values)}")
    # Adjust length if necessary to match the smaller set
    min_len = min(len(preds_avg), len(true_values))
    preds_avg = preds_avg[:min_len]
    true_values = true_values[:min_len]

# 5. Calculate Correlation (Pearson R)
pearson, pearson_p = pearsonr(true_values, preds_avg)
spearman, spearman_p = spearmanr(true_values, preds_avg)

# 6. Plotting
plt.figure(figsize=(8, 6))
plt.scatter(true_values, preds_avg, alpha=0.5, color='blue', s=10)

# Add a diagonal identity line (y=x) for reference
lims = [
    np.min([plt.xlim(), plt.ylim()]),  # min of both axes
    np.max([plt.xlim(), plt.ylim()]),  # max of both axes
]
plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='y=x')

plt.xlabel('True Values (Bed File Column 5)')
plt.ylabel('Enformer Predicted Average (Track 130)')
plt.title(f'Enformer vs True Data (Pearson: {pearson:.3f}; Spearman: {spearman:.3f})')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

,index,genome,identifier,file,clip,scale,sum_stat,description
101,5414,1,ENCFF745RFR,/home/drk/tillage/datasets/mouse/atac/encode/E...,32,2,mean,ATAC:C57BL/6 hindbrain embryo (11.5 days)
102,5415,1,ENCFF446XWP,/home/drk/tillage/datasets/mouse/atac/encode/E...,32,2,mean,ATAC:C57BL/6 kidney embryo (15.5 days)
103,5416,1,ENCFF700IKY,/home/drk/tillage/datasets/mouse/atac/encode/E...,32,2,mean,ATAC:C57BL/6 megakaryocyte-erythroid progenito...
104,5417,1,ENCFF769BOR,/home/drk/tillage/datasets/mouse/atac/encode/E...,32,2,mean,ATAC:C57BL/6 intestine postnatal (0 days)
105,5418,1,ENCFF860QUH,/home/drk/tillage/datasets/mouse/atac/encode/E...,32,2,mean,ATAC:C57BL/6 hindbrain embryo (12.5 days)
...,...,...,...,...,...,...,...,...
223,5536,1,UW8.1,/home/drk/tillage/datasets/mouse/atac/uw-atlas...,32,4,mean,ATAC:Cerebellar_granule_cells-clusters_8-clust...
224,5537,1,UW8.2,/home/drk/tillage/datasets/mouse/atac/uw-atlas...,32,4,mean,ATAC:Cerebellar_granule_cells-clusters_8-clust...
225,5538,1,UW9.1,/home/drk/tillage/datasets/mouse/atac/uw-atlas...,32,4,mean,ATAC:Unknown-clusters_9-cluster_1
226,5539,1,UW9.2,/home/drk/tillage/datasets/mouse/atac/uw-atlas...,32,4,mean,ATAC:Endothelial_II_cells-clusters_9-cluster_2


In [ ]:
preds_middle_4 = preds_raw[:, 446:450]

# 3. Take the average across those 4 bins
# axis=1 collapses the 4 bins into one average value
preds_avg = np.mean(preds_middle_4, axis=1).flatten()

# 3. Load the true values from the .bed file
species = "pig" # adjust as needed
bed_path = f'/home/azstephe/liverRegression/regression_liver/data/test_splits/log_test2/{species}_liver_TEST_500bp.bed'
true_values = pd.read_csv(bed_path, header=None, delim_whitespace=True).iloc[:, 4].values

# 4. Verify the 1-to-1 ratio
if len(preds_avg) != len(true_values):
    print(f"Warning: Size mismatch! Preds: {len(preds_avg)}, True: {len(true_values)}")
    # Adjust length if necessary to match the smaller set
    min_len = min(len(preds_avg), len(true_values))
    preds_avg = preds_avg[:min_len]
    true_values = true_values[:min_len]

# 5. Calculate Correlation (Pearson R)
pearson, pearson_p = pearsonr(true_values, preds_avg)
spearman, spearman_p = spearmanr(true_values, preds_avg)

# 6. Plotting
plt.figure(figsize=(8, 6))
plt.scatter(true_values, preds_avg, alpha=0.5, color='blue', s=10)

# Add a diagonal identity line (y=x) for reference
lims = [
    np.min([plt.xlim(), plt.ylim()]),  # min of both axes
    np.max([plt.xlim(), plt.ylim()]),  # max of both axes
]
plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='y=x')

plt.xlabel('True Values (Bed File Column 5)')
plt.ylabel('Enformer Predicted Average (Track 130)')
plt.title(f'Enformer vs True Data (Pearson: {pearson:.3f}; Spearman: {spearman:.3f})')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
preds_middle_4 = preds_raw[:, 446:450]

# 3. Take the average across those 4 bins
# axis=1 collapses the 4 bins into one average value
preds_avg = np.max(preds_middle_4, axis=1).flatten()

# 3. Load the true values from the .bed file
species = "pig" # adjust as needed
bed_path = f'/home/azstephe/liverRegression/regression_liver/data/test_splits/log_test2/{species}_liver_TEST_500bp.bed'
true_values = pd.read_csv(bed_path, header=None, delim_whitespace=True).iloc[:, 4].values

# 4. Verify the 1-to-1 ratio
if len(preds_avg) != len(true_values):
    print(f"Warning: Size mismatch! Preds: {len(preds_avg)}, True: {len(true_values)}")
    # Adjust length if necessary to match the smaller set
    min_len = min(len(preds_avg), len(true_values))
    preds_avg = preds_avg[:min_len]
    true_values = true_values[:min_len]

# 5. Calculate Correlation (Pearson R)
pearson, pearson_p = pearsonr(true_values, preds_avg)
spearman, spearman_p = spearmanr(true_values, preds_avg)

# 6. Plotting
plt.figure(figsize=(8, 6))
plt.scatter(true_values, preds_avg, alpha=0.5, color='blue', s=10)

# Add a diagonal identity line (y=x) for reference
lims = [
    np.min([plt.xlim(), plt.ylim()]),  # min of both axes
    np.max([plt.xlim(), plt.ylim()]),  # max of both axes
]
plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='y=x')

plt.xlabel('True Values (Bed File Column 5)')
plt.ylabel('Enformer Predicted Average (Track 130)')
plt.title(f'Enformer vs True Data (Pearson: {pearson:.3f}; Spearman: {spearman:.3f})')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()